# V78 - Name-only mentions

Pre-registered refinement: v56 mentions mixed module names with long syllabus text, possibly diluting direct intent. Name-only keywords test whether explicit module naming alone carries cleaner signal. Same protocol, fresh seed, same locks.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 5150
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v78")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Name-only vs full mention sets

Both feed identical Ridge heads under identical folds. Only the keyword source differs.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"])
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
low = [d.lower() for d in docs]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
STOP = set("dengan yang dan di ke dari untuk pada adalah ini itu sebagai dalam juga akan ada".split())

def kw(source):
    return {k for k in set(re.findall(r"[a-z]+", source.lower())) - STOP if len(k) > 3}

MN_name = np.zeros((len(all_users), len(MODS)))
MN_full = np.zeros((len(all_users), len(MODS)))
for j, m in enumerate(MODS):
    row = catalog.set_index("module_id").loc[m] if m in set(catalog["module_id"]) else None
    kn = kw(str(row["module_name"])) if row is not None else set()
    kf = kw(str(row["module_name"]) + " " + str(row["description_and_syllabus"])) if row is not None else set()
    for i, d in enumerate(low):
        MN_name[i, j] = sum(d.count(k) for k in kn)
        MN_full[i, j] = sum(d.count(k) for k in kf)

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(Xtr, alpha=1.0):
    oof = np.zeros_like(Y)
    fids = np.zeros(n_train, int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
        fids[va] = fi + 1
    return oof, fids

oof_w, fid = ridge_oof(W[:n_train])
oof_n, _ = ridge_oof((MN_name[:n_train] - MN_name[:n_train].mean(0)) / (MN_name[:n_train].std(0) + 1e-9))
oof_f, _ = ridge_oof((MN_full[:n_train] - MN_full[:n_train].mean(0)) / (MN_full[:n_train].std(0) + 1e-9))
base = ndcg5(Y, oof_w)
print(f"word={base.mean():.5f} name-only={ndcg5(Y, oof_n).mean():.5f} full-mentions={ndcg5(Y, oof_f).mean():.5f}")
combo = np.hstack([W[:n_train], (MN_name[:n_train] - MN_name[:n_train].mean(0)) / (MN_name[:n_train].std(0) + 1e-9)])
oof_c, _ = ridge_oof(combo)
d = ndcg5(Y, oof_c) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fid == k], oof_c[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"word+name-only: delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f} {'ADOPT' if (lo > GATE_FLOOR and wf >= 0) else 'reject'}")